# 🔬 crdt-merge v0.5.0 — A100 Feature Tests

Targeted stress tests for **v0.5.0 features** on NVIDIA A100 GPUs:
- **Wire format** — `serialize`/`deserialize` benchmarks at progressive scales
- **Wire compression** — `compress=True` vs `compress=False` overhead
- **Probabilistic at scale** — `MergeableHLL`, `MergeableBloom`, `MergeableCMS` up to 10M items
- **Probabilistic CRDT properties** — commutativity, associativity, idempotency verification
- **System sanity** — quick end-to-end validation of all prior features

Expected runtime: **~15–20 minutes** on A100.

Copyright 2026 Ryan Gillespie — Apache-2.0

In [ ]:
# Install dependencies
!pip install -q crdt-merge==0.5.0 psutil

import crdt_merge
print(f"crdt-merge version: {crdt_merge.__version__}")
assert crdt_merge.__version__.startswith("0.5"), f"Expected 0.5.x, got {crdt_merge.__version__}"

import torch, psutil, platform, os
print(f"Python: {platform.python_version()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"CPUs: {os.cpu_count()}")

In [ ]:
import time, gc, json, tracemalloc, random, string, os
import pandas as pd
import psutil

RESULTS = []
NOTEBOOK_VERSION = "0.5.0"

def record(suite, test, rows=0, duration_s=0.0, throughput=0.0, unit="rows/s",
           mem_mb=0.0, extra=None):
    """Append a result record."""
    entry = {
        "suite": suite,
        "test": test,
        "rows": rows,
        "duration_s": round(duration_s, 4),
        "throughput": round(throughput, 2),
        "unit": unit,
        "mem_mb": round(mem_mb, 2),
        "extra": extra or {},
    }
    RESULTS.append(entry)
    print(f"  ✓ {suite}/{test}: {throughput:,.0f} {unit} | {duration_s:.2f}s | {mem_mb:.1f} MB")

class MemTracker:
    """Context manager for peak memory tracking via tracemalloc."""
    def __enter__(self):
        gc.collect()
        tracemalloc.start()
        self._snap = tracemalloc.take_snapshot()
        return self
    def __exit__(self, *args):
        _, self.peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
    @property
    def peak_mb(self):
        return self.peak / 1e6

def gen_df(n, prefix="A", id_start=0):
    """Generate a DataFrame with n rows."""
    return pd.DataFrame({
        "id": [f"{prefix}_{i}" for i in range(id_start, id_start + n)],
        "value": [random.randint(0, 1_000_000) for _ in range(n)],
        "ts": [float(i) for i in range(n)],
    })

def gen_conflict_pair(n, overlap=0.5):
    """Generate two DataFrames with specified overlap."""
    shared = int(n * overlap)
    only_a = n - shared
    only_b = n - shared
    ids_shared = [f"S_{i}" for i in range(shared)]
    ids_a = [f"A_{i}" for i in range(only_a)]
    ids_b = [f"B_{i}" for i in range(only_b)]
    df_a = pd.DataFrame({
        "id": ids_shared + ids_a,
        "value": [random.randint(0, 1_000_000) for _ in range(shared + only_a)],
        "ts": [float(i) for i in range(shared + only_a)],
    })
    df_b = pd.DataFrame({
        "id": ids_shared + ids_b,
        "value": [random.randint(0, 1_000_000) for _ in range(shared + only_b)],
        "ts": [float(i + 1000) for i in range(shared + only_b)],
    })
    return df_a, df_b

def gen_sorted_stream(n, prefix="S"):
    """Generate a sorted iterable of dicts for streaming tests."""
    for i in range(n):
        yield {"id": f"{prefix}_{i:010d}", "value": random.randint(0, 1_000_000), "ts": float(i)}

def check_ram(required_gb=10):
    avail = psutil.virtual_memory().available / 1e9
    print(f"  Available RAM: {avail:.1f} GB (need {required_gb} GB)")
    return avail >= required_gb

print("Infrastructure ready.")

## Suite 1: WIRE_FORMAT — Serialization Benchmarks

In [ ]:
from crdt_merge import (
    serialize, deserialize, serialize_batch, deserialize_batch,
    peek_type, wire_size,
    GCounter, PNCounter, ORSet, LWWRegister, LWWMap
)

print("="*70)
print("SUITE 1: WIRE_FORMAT")
print("="*70)

# --- GCounter at progressive node counts ---
print("\n--- GCounter ---")
for num_nodes in [100, 1000, 10000]:
    gc.collect()
    c = GCounter(node_id="node_0")
    for i in range(num_nodes):
        c.increment(f"node_{i}", amount=random.randint(1, 100))

    # Serialize
    t0 = time.time()
    iterations = max(1, 10000 // num_nodes)
    for _ in range(iterations):
        data = serialize(c)
    ser_dur = (time.time() - t0) / iterations
    ser_ops = 1.0 / ser_dur if ser_dur > 0 else 0

    # Deserialize
    t0 = time.time()
    for _ in range(iterations):
        obj = deserialize(data)
    deser_dur = (time.time() - t0) / iterations
    deser_ops = 1.0 / deser_dur if deser_dur > 0 else 0

    # Verify roundtrip
    assert obj.value == c.value, f"Roundtrip failed: {obj.value} != {c.value}"

    sz = wire_size(data)
    assert peek_type(data) == "GCounter" or peek_type(data).startswith("G"), \
        f"Unexpected type: {peek_type(data)}"

    record("WIRE_FORMAT", f"GCounter_{num_nodes}_nodes",
           rows=num_nodes, duration_s=ser_dur, throughput=ser_ops, unit="ser_ops/s",
           extra={"deser_ops_s": round(deser_ops, 2), "wire_bytes": len(data),
                  "wire_size_info": sz})

# --- ORSet at progressive element counts ---
print("\n--- ORSet ---")
for num_elems in [1000, 10_000, 100_000, 500_000]:
    gc.collect()
    s = ORSet()
    for i in range(num_elems):
        s.add(f"elem_{i}")

    with MemTracker() as mt:
        t0 = time.time()
        iterations = max(1, 1000 // max(1, num_elems // 1000))
        for _ in range(iterations):
            data = serialize(s)
        ser_dur = (time.time() - t0) / iterations
    ser_ops = 1.0 / ser_dur if ser_dur > 0 else 0

    t0 = time.time()
    for _ in range(iterations):
        obj = deserialize(data)
    deser_dur = (time.time() - t0) / iterations
    deser_ops = 1.0 / deser_dur if deser_dur > 0 else 0

    # Verify roundtrip
    assert obj.contains("elem_0"), "Roundtrip verification failed"
    assert obj.contains(f"elem_{num_elems-1}"), "Roundtrip verification failed"

    record("WIRE_FORMAT", f"ORSet_{num_elems//1000}K_elems",
           rows=num_elems, duration_s=ser_dur, throughput=ser_ops, unit="ser_ops/s",
           mem_mb=mt.peak_mb,
           extra={"deser_ops_s": round(deser_ops, 2), "wire_bytes": len(data)})
    del s, obj, data
    gc.collect()

# --- LWWMap at progressive entry counts ---
print("\n--- LWWMap ---")
for num_entries in [1000, 10_000, 100_000]:
    gc.collect()
    m = LWWMap()
    for i in range(num_entries):
        m.set(f"key_{i}", f"value_{i}", timestamp=float(i), node_id="node_0")

    t0 = time.time()
    iterations = max(1, 1000 // max(1, num_entries // 1000))
    for _ in range(iterations):
        data = serialize(m)
    ser_dur = (time.time() - t0) / iterations
    ser_ops = 1.0 / ser_dur if ser_dur > 0 else 0

    t0 = time.time()
    for _ in range(iterations):
        obj = deserialize(data)
    deser_dur = (time.time() - t0) / iterations
    deser_ops = 1.0 / deser_dur if deser_dur > 0 else 0

    assert obj.get("key_0") == "value_0", "Roundtrip verification failed"

    record("WIRE_FORMAT", f"LWWMap_{num_entries//1000}K_entries",
           rows=num_entries, duration_s=ser_dur, throughput=ser_ops, unit="ser_ops/s",
           extra={"deser_ops_s": round(deser_ops, 2), "wire_bytes": len(data)})
    del m, obj, data
    gc.collect()

# --- serialize_batch ---
print("\n--- serialize_batch ---")
for batch_sz in [100, 1000, 10_000]:
    gc.collect()
    objects = []
    for i in range(batch_sz):
        c = GCounter(node_id=f"n_{i}")
        c.increment(f"n_{i}", amount=i + 1)
        objects.append(c)

    t0 = time.time()
    data = serialize_batch(objects)
    ser_dur = time.time() - t0

    t0 = time.time()
    restored = deserialize_batch(data)
    deser_dur = time.time() - t0

    assert len(restored) == batch_sz, f"Expected {batch_sz} objects, got {len(restored)}"
    assert restored[0].value == 1, f"First object value: {restored[0].value}"

    record("WIRE_FORMAT", f"batch_{batch_sz}",
           rows=batch_sz, duration_s=ser_dur,
           throughput=batch_sz / ser_dur if ser_dur > 0 else 0, unit="objs/s",
           extra={"deser_s": round(deser_dur, 4), "wire_bytes": len(data),
                  "bytes_per_obj": round(len(data) / batch_sz, 2)})
    del objects, restored, data
    gc.collect()

print("\n✅ WIRE_FORMAT suite complete.")

## Suite 2: WIRE_COMPRESSION — Compression Benchmarks

In [ ]:
from crdt_merge import serialize, deserialize, GCounter, ORSet, LWWMap

print("="*70)
print("SUITE 2: WIRE_COMPRESSION")
print("="*70)

def build_crdt(crdt_type, size):
    """Build a CRDT of given type and size."""
    if crdt_type == "GCounter":
        c = GCounter(node_id="node_0")
        for i in range(size):
            c.increment(f"node_{i}", amount=random.randint(1, 1000))
        return c
    elif crdt_type == "ORSet":
        s = ORSet()
        for i in range(size):
            s.add(f"element_{i}")
        return s
    elif crdt_type == "LWWMap":
        m = LWWMap()
        for i in range(size):
            m.set(f"key_{i}", f"value_{i}_" + "x" * 50, timestamp=float(i), node_id="n0")
        return m

test_configs = [
    ("GCounter", 100),
    ("GCounter", 1000),
    ("GCounter", 10000),
    ("ORSet", 1000),
    ("ORSet", 10000),
    ("ORSet", 100000),
    ("LWWMap", 1000),
    ("LWWMap", 10000),
    ("LWWMap", 100000),
]

for crdt_type, size in test_configs:
    gc.collect()
    obj = build_crdt(crdt_type, size)

    # Without compression
    t0 = time.time()
    data_raw = serialize(obj, compress=False)
    dur_raw = time.time() - t0

    # With compression
    t0 = time.time()
    data_comp = serialize(obj, compress=True)
    dur_comp = time.time() - t0

    # Verify roundtrip with compression
    restored = deserialize(data_comp)

    raw_bytes = len(data_raw)
    comp_bytes = len(data_comp)
    ratio = raw_bytes / comp_bytes if comp_bytes > 0 else 0
    overhead_pct = ((dur_comp - dur_raw) / dur_raw * 100) if dur_raw > 0 else 0

    record("WIRE_COMPRESSION", f"{crdt_type}_{size}",
           rows=size, duration_s=dur_comp,
           throughput=ratio, unit="x_ratio",
           extra={
               "raw_bytes": raw_bytes,
               "comp_bytes": comp_bytes,
               "compression_ratio": round(ratio, 2),
               "raw_ser_s": round(dur_raw, 4),
               "comp_ser_s": round(dur_comp, 4),
               "overhead_pct": round(overhead_pct, 2),
               "savings_pct": round((1 - comp_bytes / raw_bytes) * 100, 2) if raw_bytes > 0 else 0,
           })
    del obj, data_raw, data_comp, restored
    gc.collect()

print("\n✅ WIRE_COMPRESSION suite complete.")

## Suite 3: PROBABILISTIC_SCALE — HLL, Bloom, CMS at Scale

In [ ]:
from crdt_merge import MergeableHLL, MergeableBloom, MergeableCMS
import math

print("="*70)
print("SUITE 3: PROBABILISTIC_SCALE")
print("="*70)

# ── HLL: cardinality accuracy at scale ──
print("\n--- MergeableHLL ---")
for n_items in [1_000_000, 5_000_000, 10_000_000]:
    gc.collect()
    hll = MergeableHLL(precision=14)

    t0 = time.time()
    # Add in chunks for efficiency
    chunk_size = 100_000
    for start in range(0, n_items, chunk_size):
        end = min(start + chunk_size, n_items)
        items = [f"item_{i}" for i in range(start, end)]
        hll.add_all(items)
    add_dur = time.time() - t0

    est = hll.cardinality()
    error_pct = abs(est - n_items) / n_items * 100
    std_err = hll.standard_error()
    mem = hll.size_bytes()

    record("PROB_SCALE", f"HLL_{n_items//1_000_000}M",
           rows=n_items, duration_s=add_dur,
           throughput=n_items / add_dur if add_dur > 0 else 0, unit="adds/s",
           mem_mb=mem / 1e6,
           extra={"cardinality": est, "actual": n_items,
                  "error_pct": round(error_pct, 4), "std_error": std_err})

    # Error should be within 3x standard error (very generous)
    expected_err = std_err * 100  # std_error is a fraction
    print(f"    Cardinality: {est:,.0f} (actual {n_items:,}, error {error_pct:.3f}%)")
    del hll
    gc.collect()

# ── HLL merge: 2, 5, 10 independent sketches ──
print("\n--- HLL merge ---")
for num_sketches in [2, 5, 10]:
    gc.collect()
    items_per = 1_000_000
    sketches = []
    for s_idx in range(num_sketches):
        h = MergeableHLL(precision=14)
        items = [f"sketch_{s_idx}_item_{i}" for i in range(items_per)]
        h.add_all(items)
        sketches.append(h)

    t0 = time.time()
    merged = sketches[0]
    for s in sketches[1:]:
        merged = merged.merge(s)
    merge_dur = time.time() - t0

    total_unique = num_sketches * items_per
    est = merged.cardinality()
    error_pct = abs(est - total_unique) / total_unique * 100

    record("PROB_SCALE", f"HLL_merge_{num_sketches}",
           rows=total_unique, duration_s=merge_dur,
           throughput=num_sketches / merge_dur if merge_dur > 0 else 0, unit="merges/s",
           extra={"cardinality": est, "actual": total_unique,
                  "error_pct": round(error_pct, 4), "num_sketches": num_sketches})
    del sketches, merged
    gc.collect()

# ── Bloom: false positive rate at scale ──
print("\n--- MergeableBloom ---")
gc.collect()
bloom = MergeableBloom(capacity=1_000_000, fp_rate=0.01)

# Add 1M items
t0 = time.time()
chunk_size = 100_000
for start in range(0, 1_000_000, chunk_size):
    end = min(start + chunk_size, 1_000_000)
    items = [f"bloom_item_{i}" for i in range(start, end)]
    bloom.add_all(items)
add_dur = time.time() - t0

# Test false positive rate with 10M lookups
t0 = time.time()
fp_count = 0
test_items = 10_000_000
chunk_size = 500_000
for start in range(0, test_items, chunk_size):
    end = min(start + chunk_size, test_items)
    for i in range(start, end):
        # These items were NOT added (different prefix)
        if bloom.contains(f"test_fp_{i}"):
            fp_count += 1
lookup_dur = time.time() - t0
actual_fp_rate = fp_count / test_items

record("PROB_SCALE", "Bloom_1M_cap",
       rows=1_000_000, duration_s=add_dur,
       throughput=1_000_000 / add_dur if add_dur > 0 else 0, unit="adds/s",
       mem_mb=bloom.size_bytes() / 1e6,
       extra={"fp_count": fp_count, "fp_rate_actual": round(actual_fp_rate, 6),
              "fp_rate_target": 0.01, "fp_rate_estimated": bloom.estimated_fp_rate(),
              "lookup_dur_s": round(lookup_dur, 2),
              "lookups_per_s": round(test_items / lookup_dur, 2) if lookup_dur > 0 else 0})
print(f"    FP rate: {actual_fp_rate:.5f} (target: 0.01)")

# Bloom merge
print("\n--- Bloom merge ---")
for num_sketches in [2, 5, 10]:
    gc.collect()
    blooms = []
    for s_idx in range(num_sketches):
        b = MergeableBloom(capacity=100_000, fp_rate=0.01)
        items = [f"bm_{s_idx}_{i}" for i in range(100_000)]
        b.add_all(items)
        blooms.append(b)

    t0 = time.time()
    merged = blooms[0]
    for b in blooms[1:]:
        merged = merged.merge(b)
    merge_dur = time.time() - t0

    # Verify: known items should still be present
    assert merged.contains("bm_0_0"), "Merge lost items"
    assert merged.contains(f"bm_{num_sketches-1}_0"), "Merge lost items"

    record("PROB_SCALE", f"Bloom_merge_{num_sketches}",
           rows=num_sketches * 100_000, duration_s=merge_dur,
           throughput=num_sketches / merge_dur if merge_dur > 0 else 0, unit="merges/s",
           extra={"num_sketches": num_sketches})
    del blooms, merged
    gc.collect()

# ── CMS: frequency estimation with Zipf distribution ──
print("\n--- MergeableCMS ---")
gc.collect()
cms = MergeableCMS(width=2000, depth=7)

# Generate Zipf distribution (1M items)
import numpy as np
n_cms = 1_000_000
zipf_data = np.random.zipf(a=1.5, size=n_cms)
zipf_items = [f"cms_item_{v}" for v in zipf_data]

t0 = time.time()
cms.add_all(zipf_items)
add_dur = time.time() - t0

# Check frequency estimates for top items
from collections import Counter
true_counts = Counter(zipf_items)
top_items = true_counts.most_common(20)

errors = []
for item, true_count in top_items:
    est_count = cms.estimate(item)
    # CMS overestimates, never underestimates
    assert est_count >= true_count, \
        f"CMS underestimate: {est_count} < {true_count} for {item}"
    errors.append(est_count - true_count)

avg_overest = sum(errors) / len(errors) if errors else 0

record("PROB_SCALE", "CMS_1M_zipf",
       rows=n_cms, duration_s=add_dur,
       throughput=n_cms / add_dur if add_dur > 0 else 0, unit="adds/s",
       mem_mb=cms.size_bytes() / 1e6,
       extra={"avg_overestimate": round(avg_overest, 2),
              "top_item_true": top_items[0][1],
              "top_item_est": cms.estimate(top_items[0][0])})

# CMS merge
print("\n--- CMS merge ---")
for num_sketches in [2, 5, 10]:
    gc.collect()
    sketches = []
    for s_idx in range(num_sketches):
        c = MergeableCMS(width=2000, depth=7)
        items = [f"cms_m_{s_idx}_{i}" for i in range(100_000)]
        c.add_all(items)
        sketches.append(c)

    t0 = time.time()
    merged = sketches[0]
    for c in sketches[1:]:
        merged = merged.merge(c)
    merge_dur = time.time() - t0

    # Verify merged sketch has reasonable estimates
    est = merged.estimate("cms_m_0_0")
    assert est >= 1, f"Merged CMS lost data: estimate={est}"

    record("PROB_SCALE", f"CMS_merge_{num_sketches}",
           rows=num_sketches * 100_000, duration_s=merge_dur,
           throughput=num_sketches / merge_dur if merge_dur > 0 else 0, unit="merges/s",
           extra={"num_sketches": num_sketches})
    del sketches, merged
    gc.collect()

print("\n✅ PROBABILISTIC_SCALE suite complete.")

## Suite 4: PROBABILISTIC_CRDT_PROPERTIES — CRDT Verification

In [ ]:
from crdt_merge import MergeableHLL, MergeableBloom, MergeableCMS

print("="*70)
print("SUITE 4: PROBABILISTIC_CRDT_PROPERTIES")
print("="*70)

N_TRIALS = 500

def random_items(n):
    return [f"item_{random.randint(0, 1_000_000)}" for _ in range(n)]

# ── HLL properties ──
print("\n--- HLL CRDT Properties ---")
t0 = time.time()
for trial in range(N_TRIALS):
    n_items = random.randint(100, 10_000)
    items_a = random_items(n_items)
    items_b = random_items(n_items)
    items_c = random_items(n_items)

    a = MergeableHLL(precision=14)
    a.add_all(items_a)
    b = MergeableHLL(precision=14)
    b.add_all(items_b)
    c = MergeableHLL(precision=14)
    c.add_all(items_c)

    # Commutativity: a.merge(b) == b.merge(a)
    ab = a.merge(b)
    ba = b.merge(a)
    assert ab.cardinality() == ba.cardinality(), \
        f"HLL commutativity failed at trial {trial}"

    # Associativity: (a.merge(b)).merge(c) == a.merge(b.merge(c))
    ab_c = ab.merge(c)
    bc = b.merge(c)
    a_bc = a.merge(bc)
    assert ab_c.cardinality() == a_bc.cardinality(), \
        f"HLL associativity failed at trial {trial}"

    # Idempotency: a.merge(a) == a
    aa = a.merge(a)
    assert aa.cardinality() == a.cardinality(), \
        f"HLL idempotency failed at trial {trial}"

hll_dur = time.time() - t0
record("PROB_CRDT", "HLL_properties",
       rows=N_TRIALS, duration_s=hll_dur,
       throughput=N_TRIALS / hll_dur if hll_dur > 0 else 0, unit="trials/s")

# ── Bloom properties ──
print("\n--- Bloom CRDT Properties ---")
t0 = time.time()
for trial in range(N_TRIALS):
    n_items = random.randint(100, 5000)
    cap = max(n_items * 2, 1000)
    items_a = random_items(n_items)
    items_b = random_items(n_items)
    items_c = random_items(n_items)

    a = MergeableBloom(capacity=cap, fp_rate=0.01)
    a.add_all(items_a)
    b = MergeableBloom(capacity=cap, fp_rate=0.01)
    b.add_all(items_b)
    c = MergeableBloom(capacity=cap, fp_rate=0.01)
    c.add_all(items_c)

    # Commutativity: check all items from a present in both merge orders
    ab = a.merge(b)
    ba = b.merge(a)
    for item in items_a[:10]:
        assert ab.contains(item) == ba.contains(item), \
            f"Bloom commutativity failed at trial {trial}"

    # Associativity
    ab_c = ab.merge(c)
    bc = b.merge(c)
    a_bc = a.merge(bc)
    for item in items_a[:10] + items_b[:10] + items_c[:10]:
        assert ab_c.contains(item) == a_bc.contains(item), \
            f"Bloom associativity failed at trial {trial}"

    # Idempotency
    aa = a.merge(a)
    for item in items_a[:10]:
        assert aa.contains(item) == a.contains(item), \
            f"Bloom idempotency failed at trial {trial}"

bloom_dur = time.time() - t0
record("PROB_CRDT", "Bloom_properties",
       rows=N_TRIALS, duration_s=bloom_dur,
       throughput=N_TRIALS / bloom_dur if bloom_dur > 0 else 0, unit="trials/s")

# ── CMS properties ──
print("\n--- CMS CRDT Properties ---")
t0 = time.time()
for trial in range(N_TRIALS):
    n_items = random.randint(100, 5000)
    items_a = random_items(n_items)
    items_b = random_items(n_items)
    items_c = random_items(n_items)

    a = MergeableCMS(width=2000, depth=7)
    a.add_all(items_a)
    b = MergeableCMS(width=2000, depth=7)
    b.add_all(items_b)
    c = MergeableCMS(width=2000, depth=7)
    c.add_all(items_c)

    test_items_list = list(set(items_a[:5] + items_b[:5]))

    # Commutativity
    ab = a.merge(b)
    ba = b.merge(a)
    for item in test_items_list:
        assert ab.estimate(item) == ba.estimate(item), \
            f"CMS commutativity failed at trial {trial}"

    # Associativity
    ab_c = ab.merge(c)
    bc = b.merge(c)
    a_bc = a.merge(bc)
    for item in test_items_list:
        assert ab_c.estimate(item) == a_bc.estimate(item), \
            f"CMS associativity failed at trial {trial}"

    # Idempotency
    aa = a.merge(a)
    for item in test_items_list:
        # For CMS, merge(self) doubles counts — idempotency means merge(self) == 2*self
        # Actually, CMS merge takes max of counts, so it should be idempotent
        assert aa.estimate(item) == a.estimate(item), \
            f"CMS idempotency failed at trial {trial}: {aa.estimate(item)} != {a.estimate(item)}"

cms_dur = time.time() - t0
record("PROB_CRDT", "CMS_properties",
       rows=N_TRIALS, duration_s=cms_dur,
       throughput=N_TRIALS / cms_dur if cms_dur > 0 else 0, unit="trials/s")

print("\n✅ PROBABILISTIC_CRDT_PROPERTIES suite complete.")

## Suite 5: SYSTEM_SANITY — Quick End-to-End Validation

In [ ]:
from crdt_merge import (
    merge, merge_with_provenance, merge_sorted_stream, StreamStats,
    verify_crdt, verify_convergence, compute_delta, apply_delta
)

print("="*70)
print("SUITE 5: SYSTEM_SANITY")
print("="*70)

N_SANITY = 500_000

def merge_fn(a, b):
    return merge(a, b, key="id")

def gen_fn():
    n = random.randint(10, 200)
    return gen_conflict_pair(n, overlap=random.uniform(0.2, 0.8))

# 5a: Core merge at 500K
print("\n--- 5a: Core merge ---")
df_a, df_b = gen_conflict_pair(N_SANITY, overlap=0.5)
gc.collect()
t0 = time.time()
result = merge(df_a, df_b, key="id")
dur = time.time() - t0
tput = N_SANITY / dur if dur > 0 else 0
record("SANITY", "core_merge_500K", rows=N_SANITY, duration_s=dur, throughput=tput, unit="rows/s")
del result
gc.collect()

# 5b: Strategies at 500K
print("\n--- 5b: Strategies ---")
for strategy in ["lww", "max", "min"]:
    gc.collect()
    t0 = time.time()
    result = merge(df_a, df_b, key="id", strategy=strategy, timestamp_col="ts")
    dur = time.time() - t0
    tput = N_SANITY / dur if dur > 0 else 0
    record("SANITY", f"strategy_{strategy}_500K", rows=N_SANITY, duration_s=dur,
           throughput=tput, unit="rows/s")
    del result
    gc.collect()

del df_a, df_b
gc.collect()

# 5c: Sorted stream at 1M
print("\n--- 5c: Sorted stream at 1M ---")
stats = StreamStats()
gc.collect()
with MemTracker() as mt:
    t0 = time.time()
    row_count = 0
    for batch in merge_sorted_stream(
        gen_sorted_stream(1_000_000, "A"),
        gen_sorted_stream(1_000_000, "B"),
        key="id", batch_size=5000, stats=stats):
        row_count += len(batch)
    dur = time.time() - t0
record("SANITY", "stream_1M", rows=row_count, duration_s=dur,
       throughput=row_count / dur if dur > 0 else 0, unit="rows/s",
       mem_mb=mt.peak_mb)

# 5d: Provenance at 100K
print("\n--- 5d: Provenance at 100K ---")
df_a, df_b = gen_conflict_pair(100_000, overlap=0.5)
gc.collect()
t0 = time.time()
prov_result, prov_log = merge_with_provenance(df_a, df_b, key="id", timestamp_col="ts")
dur = time.time() - t0
record("SANITY", "provenance_100K", rows=100_000, duration_s=dur,
       throughput=100_000 / dur if dur > 0 else 0, unit="rows/s",
       extra={"conflicts": prov_log.total_conflicts})
del df_a, df_b, prov_result, prov_log
gc.collect()

# 5e: CRDT properties at 500 trials
print("\n--- 5e: CRDT properties ---")
t0 = time.time()
vr = verify_crdt(merge_fn, gen_fn, trials=500, include_convergence=True)
dur = time.time() - t0
record("SANITY", "crdt_properties_500", rows=500, duration_s=dur,
       throughput=round(500 / dur, 2) if dur > 0 else 0, unit="trials/s")

# 5f: 5-replica convergence at 100K
print("\n--- 5f: 5-replica convergence ---")
def gen_fn_100k():
    return gen_conflict_pair(100_000, overlap=0.5)

t0 = time.time()
vr = verify_convergence(merge_fn, gen_fn_100k, trials=5, num_replicas=5)
dur = time.time() - t0
record("SANITY", "convergence_5rep_100K", rows=100_000, duration_s=dur,
       throughput=round(5 / dur, 2) if dur > 0 else 0, unit="trials/s")

print("\n✅ SYSTEM_SANITY suite complete.")

## Results Summary

In [ ]:
import json
from datetime import datetime, timezone

print("="*70)
print("RESULTS SUMMARY — crdt-merge v0.5.0 A100 Feature Tests")
print("="*70)

# Group by suite
suites = {}
for r in RESULTS:
    suites.setdefault(r["suite"], []).append(r)

for suite, tests in suites.items():
    print(f"\n{'─'*60}")
    print(f"  {suite}")
    print(f"{'─'*60}")
    for t in tests:
        extra_str = ""
        if t["extra"]:
            highlights = {k: v for k, v in t["extra"].items()
                         if k in ("compression_ratio", "error_pct", "fp_rate_actual",
                                  "wire_bytes", "num_sketches", "overhead_pct")}
            if highlights:
                extra_str = " | " + ", ".join(f"{k}={v}" for k, v in highlights.items())
        print(f"  {t['test']:40s} {t['throughput']:>12,.0f} {t['unit']:10s} "
              f"{t['duration_s']:>8.2f}s  {t['mem_mb']:>7.1f} MB{extra_str}")

# Export JSON
report = {
    "notebook": "crdt-merge v0.5.0 A100 Feature Tests",
    "version": NOTEBOOK_VERSION,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "results": RESULTS,
    "total_tests": len(RESULTS),
    "suites": list(suites.keys()),
}

json_path = "crdt_merge_v050_a100_results.json"
with open(json_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\n📁 Results saved to {json_path}")

# Try Google Drive save
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_path = "/content/drive/MyDrive/crdt-merge-benchmarks/"
    os.makedirs(drive_path, exist_ok=True)
    import shutil
    shutil.copy(json_path, drive_path + json_path)
    print(f"📁 Also saved to Google Drive: {drive_path}{json_path}")
except Exception as e:
    print(f"ℹ️  Google Drive save skipped: {e}")

print(f"\n🏁 All {len(RESULTS)} tests complete.")